# Celeb-DF + LLM pipeline trên Google Colab (P0 → P2)

Notebook chạy các script trong `RUN_GUIDE_CELEBDF.md` trên Colab.

**Trước khi chạy:**
1. *Runtime → Change runtime type → GPU* (T4 bản free là đủ cho P0, P2 và train; annotate MLLM trên T4 cần `--load-4bit`).
2. Đặt Celeb-DF v2 lên Google Drive, nên để dạng **file zip** (copy 1 file zip nhanh hơn rất nhiều so với hàng nghìn file nhỏ).
3. Colab free giới hạn khoảng 12 giờ/phiên và ngắt khi để không. Mọi kết quả quan trọng (manifest, checkpoint, csv) được ghi vào Drive, và các bước đều chạy lại/resume được.

**Chế độ chạy:** `QUICK = True` dùng một phần nhỏ dữ liệu để kiểm tra pipeline (khoảng 1 giờ). Đặt `QUICK = False` để chạy đầy đủ (nhiều phiên Colab).

In [ ]:
# ==== THAM SỐ — sửa cho đúng Drive của bạn ====
QUICK = True                                   # True: chạy thử nhỏ; False: chạy đầy đủ
CELEBDF_ZIP = "/content/drive/MyDrive/datasets/Celeb-DF-v2.zip"   # hoặc để None nếu đã giải nén sẵn trên Drive
CELEBDF_DIR_ON_DRIVE = None                    # ví dụ "/content/drive/MyDrive/datasets/Celeb-DF-v2" nếu không dùng zip
DRIVE_OUT = "/content/drive/MyDrive/celebdf_llm"  # nơi lưu kết quả bền vững
REPO_URL = "https://github.com/Thinh1124/efficientNetb4_es.git"
BRANCH = "claude/busy-noether-c0wtai"

LOCAL = "/content/work"                        # đĩa local của Colab (nhanh, mất khi hết phiên)
FRAMES_REAL, FRAMES_FAKE = (8, 4) if QUICK else (64, 32)
LIMIT_PER_FOLDER = 40 if QUICK else 0          # chỉ lấy N video mỗi thư mục khi chạy thử

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os; os.makedirs(DRIVE_OUT, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

## 1. Lấy code và cài thư viện
Nếu repo để private, dùng `https://<TOKEN>@github.com/...` (tạo fine-grained token chỉ có quyền đọc repo này) hoặc upload bản zip của repo lên Drive rồi giải nén.

In [ ]:
%cd /content
!rm -rf efficientNetb4_es && git clone -q -b {BRANCH} {REPO_URL} efficientNetb4_es
%cd /content/efficientNetb4_es
# torch/torchvision có sẵn trên Colab
!pip -q install -r requirements.txt
!pip -q install insightface onnxruntime-gpu transformers accelerate bitsandbytes

## 2. Dữ liệu Celeb-DF lên đĩa local

In [ ]:
import os, subprocess
if CELEBDF_ZIP:
    !cp "{CELEBDF_ZIP}" /content/celebdf.zip && unzip -q -o /content/celebdf.zip -d /content/celebdf_raw && rm /content/celebdf.zip
    # tìm thư mục chứa List_of_testing_videos.txt
    CELEBDF_ROOT = subprocess.check_output("dirname $(find /content/celebdf_raw -name List_of_testing_videos.txt | head -1)", shell=True, text=True).strip()
else:
    CELEBDF_ROOT = CELEBDF_DIR_ON_DRIVE
print("CELEBDF_ROOT =", CELEBDF_ROOT); print(os.listdir(CELEBDF_ROOT))
DATA = f"{LOCAL}/celebdf"; os.makedirs(DATA, exist_ok=True)

## 3. P0 — split, trích frame, manifest
`extract` resume được: nếu Colab ngắt, chạy lại ô này, các video đã xong sẽ được bỏ qua (miễn là thư mục `frames` còn). Để không mất frame khi hết phiên, ô cuối của mục này nén `frames` lên Drive.

In [ ]:
!python scripts/prepare_celebdf.py split --celebdf-root "{CELEBDF_ROOT}" --out {DATA}/splits.json --sid-heldout 12
!python scripts/prepare_celebdf.py extract --celebdf-root "{CELEBDF_ROOT}" --splits {DATA}/splits.json --frames-root {DATA}/frames --frames-real {FRAMES_REAL} --frames-fake {FRAMES_FAKE} --limit-per-folder {LIMIT_PER_FOLDER} --ctx-id 0
!python scripts/prepare_celebdf.py merge --splits {DATA}/splits.json --frames-root {DATA}/frames --out {DATA}/meta.csv
!cp {DATA}/splits.json {DATA}/meta.csv {DRIVE_OUT}/

In [ ]:
# Xem nhanh vài crop
import pandas as pd, cv2, numpy as np, matplotlib.pyplot as plt
meta = pd.read_csv(f"{DATA}/meta.csv"); sample = meta.sample(8, random_state=0)
plt.figure(figsize=(16, 2.5))
for i, r in enumerate(sample.itertuples()):
    plt.subplot(1, 8, i + 1); plt.imshow(cv2.imread(f"{DATA}/frames/{r.rel_path}")[..., ::-1]); plt.title(f"label={r.label}"); plt.axis("off")

In [ ]:
# Lưu frames lên Drive (để phiên sau không phải trích lại). Khôi phục: tar -xf .../frames.tar -C {DATA}
!tar -cf {DRIVE_OUT}/frames.tar -C {DATA} frames

## 4. P1.1 — Pseudo-mask (kiểm tra luôn quy ước tên video fake)

In [ ]:
!python scripts/make_pseudo_masks.py --celebdf-root "{CELEBDF_ROOT}" --meta {DATA}/meta.csv --masks-root {DATA}/masks --out-csv {DATA}/masks.csv
# Nếu dòng cuối cảnh báo < 50% aligned: xoá masks.csv và chạy lại với --src-naming source
!cp {DATA}/masks.csv {DRIVE_OUT}/

In [ ]:
m = pd.read_csv(f"{DATA}/masks.csv"); print(m["aligned"].mean(), m["dominant_region"].value_counts().to_dict())
show = m[m.aligned].sample(min(6, int(m.aligned.sum())), random_state=0)
plt.figure(figsize=(14, 5))
for i, r in enumerate(show.itertuples()):
    plt.subplot(2, 6, i + 1); plt.imshow(cv2.imread(f"{DATA}/frames/{r.rel_path}")[..., ::-1]); plt.axis("off")
    plt.subplot(2, 6, i + 7); plt.imshow(cv2.imread(f"{DATA}/masks/{r.mask_rel}", 0), cmap="gray"); plt.title(r.dominant_region); plt.axis("off")

## 5. P0.4 — Baseline B0
Config được sinh tự động từ `configs/celebdf.yaml`. Checkpoint ghi thẳng lên Drive; nếu bị ngắt, đặt `RESUME = True` rồi chạy lại.

In [ ]:
import yaml
def make_config(name, seed=0, protocol="official", synth=None, epochs=None, **extra):
    cfg = yaml.safe_load(open("configs/celebdf.yaml"))
    cfg.update(manifest=f"{DATA}/meta.csv", frames_root=f"{DATA}/frames", protocol=protocol, seed=seed,
               synth=synth or [], num_workers=2, batch_size=32,
               epochs=epochs or (3 if QUICK else 30), save_dir=f"{DRIVE_OUT}/checkpoints/{name}", **extra)
    path = f"{LOCAL}/{name}.yaml"; yaml.safe_dump(cfg, open(path, "w")); return path, f"{cfg['save_dir']}/best.pth"

RESUME = False
cfg_b0, ckpt_b0 = make_config("B0_seed0")
!python train.py --config {cfg_b0} {"--resume " + ckpt_b0 if RESUME else ""}

In [ ]:
!python scripts/predict_csv.py --checkpoint {ckpt_b0} --manifest {DATA}/meta.csv --frames-root {DATA}/frames --split test --num-workers 2 --out {DRIVE_OUT}/results/B0_seed0/test_official.csv
!python scripts/predict_csv.py --checkpoint {ckpt_b0} --manifest {DATA}/meta.csv --frames-root {DATA}/frames --split test --num-workers 2 --perturb jpeg:30 --out {DRIVE_OUT}/results/B0_seed0/test_jpeg30.csv
!python scripts/eval_predictions.py {DRIVE_OUT}/results/B0_seed0/test_official.csv {DRIVE_OUT}/results/B0_seed0/test_jpeg30.csv

## 6. P1.2–1.3 — Annotation MLLM
Trên T4 (16 GB) dùng `--load-4bit`; script tự chọn fp16 vì T4 không hỗ trợ bf16. Chạy `--dry-run` trước để xem prompt và ảnh gợi ý.

In [ ]:
N_FAKE, N_REAL = (40, 10) if QUICK else (8000, 2000)
ANN = f"{DRIVE_OUT}/annotations/train_mllm.jsonl"   # ghi thẳng lên Drive -> resume được
!python scripts/annotate_mllm.py --meta {DATA}/meta.csv --frames-root {DATA}/frames --masks-csv {DATA}/masks.csv --masks-root {DATA}/masks --out {LOCAL}/preview/ann.jsonl --n-fake 4 --n-real 2 --dry-run
!python scripts/annotate_mllm.py --meta {DATA}/meta.csv --frames-root {DATA}/frames --masks-csv {DATA}/masks.csv --masks-root {DATA}/masks --out {ANN} --n-fake {N_FAKE} --n-real {N_REAL} --load-4bit --batch-size 4

In [ ]:
# Xuất phiếu chấm cho 2 người (tải thư mục review về, điền region_ok/artifact_ok, upload lại vào cùng chỗ)
!python scripts/qc_annotations.py sample-review --ann {ANN} --frames-root {DATA}/frames --masks-root {DATA}/masks --n 200 --out-dir {DRIVE_OUT}/review
# Sau khi đã điền xong 2 file reviewer_*.csv:
# !python scripts/qc_annotations.py calibrate --reviews {DRIVE_OUT}/review/reviewer_A.csv {DRIVE_OUT}/review/reviewer_B.csv --out {DRIVE_OUT}/annotations/calib.json
# !python scripts/qc_annotations.py score --ann {ANN} --masks-csv {DATA}/masks.csv --calib {DRIVE_OUT}/annotations/calib.json --out {DRIVE_OUT}/annotations/train_qc.jsonl

## 7. P2 — Dữ liệu tổng hợp + planner
Ảnh synth sinh trên đĩa local (CPU Colab chỉ có 2 nhân nên để `--workers 2`), sau đó nén lên Drive.

In [ ]:
SYN = f"{LOCAL}/synth"; PER_CELL = 10 if QUICK else 40; BUDGET = 2000 if QUICK else 10000
!python -m synth.generate --meta {DATA}/meta.csv --frames-root {DATA}/frames --split val --grid --per-cell {PER_CELL} --out-dir {SYN}/probe_val --workers 2
# vòng 0: D1 ngẫu nhiên (dùng chung làm điểm xuất phát cho D2/D3)
!python -m synth.generate --meta {DATA}/meta.csv --frames-root {DATA}/frames --split train --random {BUDGET} --round 0 --out-dir {SYN}/D2 --workers 2

In [ ]:
# Một vòng planner: train ngắn -> dự đoán val + probe -> báo cáo -> kế hoạch -> sinh ảnh vòng tiếp theo
ROUND = 1
cfg_r, ckpt_r = make_config(f"D2_r{ROUND}", synth=[{"manifest": f"{SYN}/D2/synth_meta.csv", "rounds": list(range(ROUND))}],
                            epochs=2 if QUICK else 5)
R = f"{DRIVE_OUT}/results/D2_r{ROUND}"
!python train.py --config {cfg_r}
!python scripts/predict_csv.py --checkpoint {ckpt_r} --manifest {DATA}/meta.csv --frames-root {DATA}/frames --split val --num-workers 2 --out {R}/val.csv
!python scripts/predict_csv.py --checkpoint {ckpt_r} --synth {SYN}/probe_val/synth_meta.csv --num-workers 2 --out {R}/probe.csv
!python -m synth.error_report --probe-preds {R}/probe.csv --val-preds {R}/val.csv --masks-csv {DATA}/masks.csv --out {R}/report

In [ ]:
# Kế hoạch bằng LLM chạy ngay trong Colab (Qwen2.5-7B-Instruct 4-bit vừa T4; muốn nhanh hơn dùng Qwen/Qwen2.5-3B-Instruct)
PLAN = f"{DRIVE_OUT}/plans/D2_r{ROUND + 1}.json"
!python -m synth.llm_planner --report {R}/report.json --mode llm --backend transformers --model Qwen/Qwen2.5-7B-Instruct --load-4bit --budget {BUDGET} --round {ROUND + 1} --out {PLAN} --log {DRIVE_OUT}/runs/planner_log.jsonl
!python -m synth.generate --meta {DATA}/meta.csv --frames-root {DATA}/frames --split train --plan {PLAN} --round {ROUND + 1} --out-dir {SYN}/D2 --workers 2
# Lặp lại 2 ô trên với ROUND = 2, 3, ...  D3: thay --mode llm bằng --mode rule (và out-dir D3). D1: --mode random.

In [ ]:
# Lưu dữ liệu synth lên Drive
!tar -cf {DRIVE_OUT}/synth.tar -C {LOCAL} synth

## Ghi chú
- Khi mở phiên mới: chạy mục 1, tải lại dữ liệu ở mục 2, rồi khôi phục `frames.tar` / `synth.tar` từ Drive vào `LOCAL` thay vì chạy lại P0/P2.
- Backend transformers của planner chưa được chạy thử với model thật (môi trường phát triển không tải được model từ Hugging Face); các bước còn lại đã chạy thử trên dữ liệu giả lập.